In [98]:
import numpy as np
import pandas as pd
import spacy
from spacy.matcher import PhraseMatcher, Matcher, DependencyMatcher
nlp = spacy.load("en_core_web_sm")

In [364]:
from itertools import combinations
from sklearn.utils import shuffle
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import joblib
import gensim

import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from nltk.tokenize import sent_tokenize, word_tokenize

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [ ]:
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    return wordnet.NOUN

In [ ]:
stop_words = set(stopwords.words("english"))
stop_words -= {"no", "nor", "not", "never",
    "none", "nothing", "nobody", "neither"}

def tokenize_text(new_df):
    new_df["text"] = new_df["text"].astype('string')

    processed_list = []
    bad_idx = []
    for idx, sentence in enumerate(new_df['text']):
        if sentence:
            words = word_tokenize(sentence)
            filtered_words = [word for word in words if word.lower() not in stop_words]

            tokens = [t for t in filtered_words if t.isalpha()]

            filtered_tokens = [t for t in tokens if t not in stop_words]

            tagged = nltk.pos_tag(filtered_tokens)

            lemmatized = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in tagged]

            text = " ".join(lemmatized)
            if text.strip() == "":
                bad_idx.append(idx)
            else:
                processed_list.append(text)
        else:
            bad_idx.append(idx)
    new_df = new_df.drop(index=bad_idx).reset_index(drop=True)

    new_df.insert(1,"processed_text", processed_list)
    new_df = new_df.drop(columns=["text"])
    return new_df

In [10]:
df = pd.read_csv("cleaned_house_repair_dataset.csv")
df.loc[df["positive"] == 1, "positive"] = 2
df.loc[df["neutral"] == 1, "neutral"] = 1
df.loc[df["negative"] == 1, "negative"] = 0
df.insert(1, "sentiment", df["negative"] + df["positive"] + df["neutral"])
df = df.drop(columns=["positive", "neutral", "negative"])

In [697]:
def get_pattern(text):
  split_text = text.split()
  length = len(split_text)
  split_text += split_text

  comb_list = set(list(combinations(split_text, length)) +(list(combinations(split_text, length-1))))
  bad_values = set()

  for comb in comb_list:
    tokens = []
    for tok in comb:
      if tok in tokens:
        bad_values.add(comb)
      else:
        tokens.append(tok)
  comb_list = list(comb_list - bad_values)

  patterns = []
  for comb in comb_list:
    if comb:
      pattern = []
      for tok in comb:
        pattern.append({
            "LEMMA": {"FUZZY": tok}
        })
      patterns.append(pattern)

  print(patterns)
  return patterns

In [698]:
df = pd.read_csv("cleaned_house_repair_dataset.csv")

matcher = Matcher(nlp.vocab)

for item in df["item"].values:
  matcher.add(item.upper().replace(" ", "_"), get_pattern(item))

[[{'LEMMA': {'FUZZY': 'shower'}}, {'LEMMA': {'FUZZY': 'install'}}], [{'LEMMA': {'FUZZY': 'install'}}, {'LEMMA': {'FUZZY': 'shower'}}], [{'LEMMA': {'FUZZY': 'shower'}}, {'LEMMA': {'FUZZY': 'install'}}, {'LEMMA': {'FUZZY': 'stall'}}], [{'LEMMA': {'FUZZY': 'install'}}, {'LEMMA': {'FUZZY': 'shower'}}, {'LEMMA': {'FUZZY': 'stall'}}], [{'LEMMA': {'FUZZY': 'install'}}, {'LEMMA': {'FUZZY': 'stall'}}], [{'LEMMA': {'FUZZY': 'shower'}}, {'LEMMA': {'FUZZY': 'stall'}}, {'LEMMA': {'FUZZY': 'install'}}], [{'LEMMA': {'FUZZY': 'stall'}}, {'LEMMA': {'FUZZY': 'install'}}, {'LEMMA': {'FUZZY': 'shower'}}], [{'LEMMA': {'FUZZY': 'shower'}}, {'LEMMA': {'FUZZY': 'stall'}}], [{'LEMMA': {'FUZZY': 'stall'}}, {'LEMMA': {'FUZZY': 'shower'}}], [{'LEMMA': {'FUZZY': 'install'}}, {'LEMMA': {'FUZZY': 'stall'}}, {'LEMMA': {'FUZZY': 'shower'}}], [{'LEMMA': {'FUZZY': 'stall'}}, {'LEMMA': {'FUZZY': 'install'}}]]
[[{'LEMMA': {'FUZZY': 'home'}}], [{'LEMMA': {'FUZZY': 'home'}}, {'LEMMA': {'FUZZY': 'automation'}}], [{'LEMMA': {

In [699]:
data = [tokenize_text(text).split() for text in df["item"].values]
word2vec_model = gensim.models.Word2Vec(data, min_count=1,
                                vector_size=200, window=10)

In [729]:
sentence = "The houses water heater has been repaired"
sentence = tokenize_text(sentence)
doc = nlp(sentence)
matches = matcher(doc)
print(sentence)

house water heater repair


In [730]:
lookup_items = []
for match_id, start, end in matches:
  string_id = nlp.vocab.strings[match_id]
  lookup_items.append(string_id.lower().replace("_", " "))

In [732]:
total_list = []
for item in lookup_items:
  similarity = 0
  for i in item.split():
    for word in sentence.split():
      try:
        similarity += word2vec_model.wv.similarity(i, word)
      except KeyError:
         similarity += 0
  total_list.append(similarity / len(item.split()))

if total_list:
  predicted_item = lookup_items[np.argmax(total_list)]
  print(predicted_item)


water heater repair
